In [1]:
import os, sys, re, json, hashlib
from pathlib import Path
from datetime import datetime

NORTHSTAR = "spam-cleaner-t39-qa"
NOTEBOOK_PATH = "notebooks/qa/58-spam-cleaner-t39-qa.ipynb"
TOWER = 39
TOWER_NAME = "Tower of Gmail Spam Cleaner"
MIN_SCORE = 70
BROWSER_ROOT = Path("/home/phuc/projects/solace-browser")
SRC = BROWSER_ROOT / "src"
WEB = BROWSER_ROOT / "web"
TESTS = BROWSER_ROOT / "tests"
results = []
def record(probe_id, name, passed, detail=""):
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status, "detail": detail})
    print(f"{status}: {name}" + (f" -- {detail}" if detail else ""))
def read_file(path):
    p = Path(path)
    return p.read_text(encoding='utf-8', errors='replace') if p.exists() else ""
print(f"Tower {TOWER}: {TOWER_NAME}")
assert BROWSER_ROOT.exists()

Tower 39: Tower of Gmail Spam Cleaner


In [2]:
# Cell 1: Spam cleaner recipe structure
print("=== Spam Cleaner Recipe ===")

apps_dir = BROWSER_ROOT / "data" / "default" / "apps"
spam_recipe = read_file(apps_dir / "gmail-spam-cleaner" / "recipe.json")
spam_data = json.loads(spam_recipe) if spam_recipe else {}

# P01: Gmail spam cleaner recipe exists
record("T39-P01", "Gmail spam cleaner recipe exists",
       '"id": "gmail-spam-cleaner"' in spam_recipe,
       "gmail-spam-cleaner/recipe.json")

# P02: Recipe has google platform with read + delete email scopes
spam_scopes = spam_data.get("oauth3_scopes", [])
record("T39-P02", "Recipe scoped to google.read.email and google.write.email.delete",
       len(spam_scopes) >= 2 and any("read" in s for s in spam_scopes) and any("delete" in s for s in spam_scopes),
       f"Scopes: {spam_scopes}")

# P03: Recipe navigates to Gmail spam folder URL
spam_steps = json.dumps(spam_data.get("steps", []))
record("T39-P03", "Recipe navigates to Gmail spam folder (#spam)",
       "#spam" in spam_steps and "mail.google.com" in spam_steps,
       "Navigates to https://mail.google.com/mail/u/0/#spam")

# P04: Recipe has auto_delete_threshold input parameter
inputs = spam_data.get("input_params", {})
record("T39-P04", "Recipe has auto_delete_threshold parameter (configurable confidence)",
       "auto_delete_threshold" in inputs,
       f"Default threshold: {inputs.get('auto_delete_threshold', {}).get('default', 'N/A')}")

# P05: Recipe extracts sender, subject, snippet, date from spam messages
record("T39-P05", "Recipe extracts sender, subject, snippet, date from spam",
       "sender" in spam_steps and "subject" in spam_steps and "snippet" in spam_steps and "date" in spam_steps,
       "Structured extraction of spam message metadata")

=== Spam Cleaner Recipe ===
PASS: Gmail spam cleaner recipe exists -- gmail-spam-cleaner/recipe.json
PASS: Recipe scoped to google.read.email and google.write.email.delete -- Scopes: ['google.read.email', 'google.write.email.delete']
PASS: Recipe navigates to Gmail spam folder (#spam) -- Navigates to https://mail.google.com/mail/u/0/#spam
PASS: Recipe has auto_delete_threshold parameter (configurable confidence) -- Default threshold: 80
PASS: Recipe extracts sender, subject, snippet, date from spam -- Structured extraction of spam message metadata


In [3]:
# Cell 2: Spam classification and report
print("=== Spam Classification & Report ===")

spam_recipe = read_file(BROWSER_ROOT / "data" / "default" / "apps" / "gmail-spam-cleaner" / "recipe.json")
spam_data = json.loads(spam_recipe) if spam_recipe else {}
spam_steps = json.dumps(spam_data.get("steps", []))

# P06: Recipe uses llm_analyze for spam classification
record("T39-P06", "Recipe uses llm_analyze for spam classification",
       "llm_analyze" in spam_steps and "spam" in spam_steps.lower(),
       "LLM classifies messages by deletion confidence")

# P07: Classification uses confidence tiers (CONFIRMED_SPAM, LIKELY_SPAM, REVIEW_NEEDED)
record("T39-P07", "Classification uses CONFIRMED_SPAM, LIKELY_SPAM, REVIEW_NEEDED tiers",
       "CONFIRMED_SPAM" in spam_steps and "LIKELY_SPAM" in spam_steps and "REVIEW_NEEDED" in spam_steps,
       "Three-tier confidence classification for safe deletion")

# P08: REVIEW_NEEDED items are explicitly marked DO NOT DELETE
record("T39-P08", "REVIEW_NEEDED items marked as DO NOT DELETE",
       "DO NOT DELETE" in spam_steps,
       "False positive protection -- review items are never auto-deleted")

# P09: Recipe generates markdown cleanup report
record("T39-P09", "Recipe generates markdown spam cleanup report",
       "generate_report" in spam_steps and "Spam Cleanup Report" in spam_steps,
       "Report shows confirmed, likely, and review counts")

# P10: Report saved to outbox before deletion (approval gate)
record("T39-P10", "Report saved to outbox before any deletion",
       "save_to_outbox" in spam_steps and "spam-cleanup" in spam_steps,
       "Output: spam-cleanup-{date}.json -- user reviews before acting")

=== Spam Classification & Report ===
PASS: Recipe uses llm_analyze for spam classification -- LLM classifies messages by deletion confidence
PASS: Classification uses CONFIRMED_SPAM, LIKELY_SPAM, REVIEW_NEEDED tiers -- Three-tier confidence classification for safe deletion
PASS: REVIEW_NEEDED items marked as DO NOT DELETE -- False positive protection -- review items are never auto-deleted
PASS: Recipe generates markdown spam cleanup report -- Report shows confirmed, likely, and review counts
PASS: Report saved to outbox before any deletion -- Output: spam-cleanup-{date}.json -- user reviews before acting


In [4]:
# Cell 3: Budget and safety constraints
print("=== Budget & Safety Constraints ===")

# P11: Spam cleaner budget file exists with cost cap
budget_content = read_file(BROWSER_ROOT / "data" / "default" / "apps" / "gmail-spam-cleaner" / "budget.json")
record("T39-P11", "Spam cleaner budget file exists with cost cap",
       "max_cost_per_run_usd" in budget_content,
       "Budget constrains LLM cost per run")

# P12: Budget limits daily LLM calls
budget_data = json.loads(budget_content) if budget_content else {}
record("T39-P12", "Budget limits daily LLM calls to 2",
       budget_data.get("daily_max_llm_calls", 0) > 0,
       f"daily_max_llm_calls={budget_data.get('daily_max_llm_calls')}")

# P13: Budget limits daily page visits
record("T39-P13", "Budget limits daily page visits",
       budget_data.get("daily_max_pages", 0) > 0,
       f"daily_max_pages={budget_data.get('daily_max_pages')}")

# P14: gmail.delete.email scope is registered as high-risk
scopes_py = read_file(SRC / "oauth3" / "scopes.py")
record("T39-P14", "gmail.delete.email scope registered as high-risk destructive",
       '"gmail.delete.email"' in scopes_py,
       "Delete email is irreversible -- requires step-up auth")

# P15: Recipe has evidence_type lane_b for destructive actions
spam_recipe = read_file(BROWSER_ROOT / "data" / "default" / "apps" / "gmail-spam-cleaner" / "recipe.json")
spam_data = json.loads(spam_recipe) if spam_recipe else {}
record("T39-P15", "Spam cleaner uses evidence_type lane_b",
       spam_data.get("evidence_type") == "lane_b",
       f"evidence_type={spam_data.get('evidence_type')} (lane_b for write operations)")

=== Budget & Safety Constraints ===
PASS: Spam cleaner budget file exists with cost cap -- Budget constrains LLM cost per run
PASS: Budget limits daily LLM calls to 2 -- daily_max_llm_calls=2
PASS: Budget limits daily page visits -- daily_max_pages=5
PASS: gmail.delete.email scope registered as high-risk destructive -- Delete email is irreversible -- requires step-up auth
PASS: Spam cleaner uses evidence_type lane_b -- evidence_type=lane_b (lane_b for write operations)


In [5]:
# Cell 4: Evidence and audit infrastructure
print("=== Evidence & Audit Infrastructure ===")

# P16: Audit chain has append-only design (no delete/modify methods)
chain_py = read_file(SRC / "audit" / "chain.py")
has_append = "def append" in chain_py
no_delete = "def delete" not in chain_py
no_modify = "def modify" not in chain_py
record("T39-P16", "Audit chain is append-only (no delete, no modify)",
       has_append and no_delete and no_modify,
       "Immutable design: append() only, no delete() or modify()")

# P17: Evidence chain validates integrity with verify_integrity
record("T39-P17", "Evidence chain has verify_integrity for tamper detection",
       "def verify_integrity" in chain_py and "TAMPER" in chain_py.upper() or "break_at" in chain_py,
       "verify_integrity() detects hash mismatches and chain breaks")

# P18: AuditEntry captures action, target, before_value, after_value
record("T39-P18", "AuditEntry captures action, target, before/after values",
       "before_value" in chain_py and "after_value" in chain_py and "action" in chain_py,
       "Per-email evidence: what changed, from what, to what")

# P19: Evidence chain has GENESIS_HASH for first entry
record("T39-P19", "Evidence chain uses GENESIS_HASH for first entry",
       'GENESIS_HASH = "0" * 64' in chain_py,
       "First entry prev_hash = 64 zeros (genesis block)")

# P20: Budget gate checker blocks when budget exhausted
budget_gates = read_file(SRC / "budget_gates.py")
record("T39-P20", "Budget gate blocks execution when budget exhausted",
       "BudgetExhaustedError" in budget_gates and "remaining_runs" in budget_gates,
       "B2 gate: remaining_runs <= 0 -> BLOCKED")

=== Evidence & Audit Infrastructure ===
PASS: Audit chain is append-only (no delete, no modify) -- Immutable design: append() only, no delete() or modify()
PASS: Evidence chain has verify_integrity for tamper detection -- verify_integrity() detects hash mismatches and chain breaks
PASS: AuditEntry captures action, target, before/after values -- Per-email evidence: what changed, from what, to what
PASS: Evidence chain uses GENESIS_HASH for first entry -- First entry prev_hash = 64 zeros (genesis block)
PASS: Budget gate blocks execution when budget exhausted -- B2 gate: remaining_runs <= 0 -> BLOCKED


In [6]:
# Cell 5: Evidence summary
passed = len([r for r in results if r["status"] == "PASS"])
failed = len([r for r in results if r["status"] == "FAIL"])
total = len(results)
score = round(passed / total * 100, 1) if total > 0 else 0
finding_rate = round(failed / total * 100, 1) if total > 0 else 0
grade = "A+" if score >= 95 else "A" if score >= 90 else "B+" if score >= 85 else "B" if score >= 80 else "C" if score >= 70 else "D" if score >= 60 else "F"
summary = {"surface": NORTHSTAR, "tower": TOWER, "tower_name": TOWER_NAME, "timestamp": datetime.now().isoformat(), "total_probes": total, "passed": passed, "failed": failed, "score": score, "grade": grade, "finding_rate": finding_rate, "converged": finding_rate < 5, "findings": [r for r in results if r["status"] == "FAIL"], "evidence_hash": hashlib.sha256(json.dumps(results, sort_keys=True).encode()).hexdigest()[:16]}
print("=" * 60)
print(f"TOWER {TOWER}: {TOWER_NAME}")
print("=" * 60)
print(f"Probes: {total} | Passed: {passed} | Failed: {failed}")
print(f"Score: {score}% | Grade: {grade} | Finding rate: {finding_rate}%")
print(f"Converged: {'YES' if summary['converged'] else 'NO'}")
print(f"Evidence hash: {summary['evidence_hash']}")
if summary["findings"]:
    print("\nFINDINGS:")
    for f in summary["findings"]:
        print(f"  [{f['id']}] {f['name']}: {f.get('detail', '')}")
print()
print(json.dumps(summary, indent=2))

TOWER 39: Tower of Gmail Spam Cleaner
Probes: 20 | Passed: 20 | Failed: 0
Score: 100.0% | Grade: A+ | Finding rate: 0.0%
Converged: YES
Evidence hash: 54a5c3de50e8a47c

{
  "surface": "spam-cleaner-t39-qa",
  "tower": 39,
  "tower_name": "Tower of Gmail Spam Cleaner",
  "timestamp": "2026-03-06T11:29:48.023230",
  "total_probes": 20,
  "passed": 20,
  "failed": 0,
  "score": 100.0,
  "grade": "A+",
  "finding_rate": 0.0,
  "converged": true,
  "findings": [],
  "evidence_hash": "54a5c3de50e8a47c"
}
